In [46]:
from entsoe import EntsoePandasClient
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv("/Users/raynapatel/Downloads/Zeus-Energy_Security_Index/datasets/entsoe/entsoe.env")
client = EntsoePandasClient(api_key="be9993ee-346c-45b9-bef1-201dd8f8bf3d")

start = pd.Timestamp("2021-01-01", tz="Europe/Berlin")
end   = pd.Timestamp("2026-05-25", tz="Europe/Berlin")

prices = client.query_day_ahead_prices("DE_LU", start=start, end=end)

os.makedirs("data/raw", exist_ok=True)
prices.to_csv("data/raw/day_ahead_prices_DE.csv")
print("Pulled successfully. Shape:", prices.shape)
print(prices.head(30))

Pulled successfully. Shape: (64267,)
2021-01-01 00:00:00+01:00    50.87
2021-01-01 01:00:00+01:00    48.19
2021-01-01 02:00:00+01:00    44.68
2021-01-01 03:00:00+01:00    42.92
2021-01-01 04:00:00+01:00    40.39
2021-01-01 05:00:00+01:00    40.20
2021-01-01 06:00:00+01:00    39.63
2021-01-01 07:00:00+01:00    40.09
2021-01-01 08:00:00+01:00    41.27
2021-01-01 09:00:00+01:00    44.88
2021-01-01 10:00:00+01:00    45.00
2021-01-01 11:00:00+01:00    47.20
2021-01-01 12:00:00+01:00    50.78
2021-01-01 13:00:00+01:00    45.49
2021-01-01 14:00:00+01:00    44.73
2021-01-01 15:00:00+01:00    46.59
2021-01-01 16:00:00+01:00    52.99
2021-01-01 17:00:00+01:00    60.26
2021-01-01 18:00:00+01:00    60.61
2021-01-01 19:00:00+01:00    60.36
2021-01-01 20:00:00+01:00    57.40
2021-01-01 21:00:00+01:00    53.86
2021-01-01 22:00:00+01:00    53.45
2021-01-01 23:00:00+01:00    49.72
2021-01-02 00:00:00+01:00    46.69
2021-01-02 01:00:00+01:00    42.43
2021-01-02 02:00:00+01:00    41.09
2021-01-02 03:00:0

In [47]:
import pandas as pd
import numpy as np
import os

#  Load 
df = pd.read_csv("data/raw/day_ahead_prices_DE.csv")
df.columns = ["timestamp", "price_eur_mwh"]

#  Parse timestamps 
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert("Europe/Berlin")
df = df.sort_values("timestamp").reset_index(drop=True)

#  Check for missing values 
print(f"Missing values: {df['price_eur_mwh'].isna().sum()}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

#  Fill small gaps 
df["price_eur_mwh"] = df["price_eur_mwh"].ffill(limit=3)

#  Add time features 
df["date"]       = df["timestamp"].dt.date
df["hour"]       = df["timestamp"].dt.hour
df["dayofweek"]  = df["timestamp"].dt.dayofweek
df["month"]      = df["timestamp"].dt.month
df["year"]       = df["timestamp"].dt.year
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

#  Daily average 
daily = df.groupby("date")["price_eur_mwh"].mean().reset_index()
daily.columns = ["date", "avg_price_eur_mwh"]
daily["date"] = pd.to_datetime(daily["date"])

#  Save 
os.makedirs("data/clean", exist_ok=True)
df.to_csv("data/clean/prices_hourly_DE.csv", index=False)
daily.to_csv("data/clean/prices_daily_DE.csv", index=False)

print("Saved. Shape:", daily.shape)
print(daily.head())

Missing values: 0
Date range: 2021-01-01 00:00:00+01:00 to 2026-05-25 00:00:00+02:00
Saved. Shape: (1971, 2)
        date  avg_price_eur_mwh
0 2021-01-01          48.398333
1 2021-01-02          50.562500
2 2021-01-03          38.622500
3 2021-01-04          48.017917
4 2021-01-05          55.337083


In [51]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import os

os.makedirs("figures", exist_ok=True)
daily = pd.read_csv("data/clean/prices_daily_DE.csv", parse_dates=["date"])
daily["month"] = daily["date"].dt.month
daily["year"]  = daily["date"].dt.year

# Plot 1: Full price history
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=daily["date"], y=daily["avg_price_eur_mwh"],
                          mode="lines", name="Price", line=dict(color="steelblue", width=1)))
fig1.add_vrect(x0="2022-02-24", x1="2022-12-31",
               fillcolor="red", opacity=0.1, line_width=0,
               annotation_text="Ukraine invasion period", annotation_position="top left")
fig1.update_layout(title="Germany Day-Ahead Electricity Prices (Daily Avg)",
                   yaxis_title="EUR/MWh", xaxis_title="Date", template="plotly_white")
fig1.write_html("figures/price_history_DE.html")
fig1.show()

# Plot 2: Seasonality by month
monthly_avg = daily.groupby("month")["avg_price_eur_mwh"].mean().reset_index()
monthly_avg["month_name"] = ["Jan","Feb","Mar","Apr","May","Jun",
                              "Jul","Aug","Sep","Oct","Nov","Dec"]
fig2 = px.bar(monthly_avg, x="month_name", y="avg_price_eur_mwh",
              title="Average Electricity Price by Month (Germany)",
              labels={"avg_price_eur_mwh": "EUR/MWh", "month_name": "Month"},
              color_discrete_sequence=["steelblue"], template="plotly_white")
fig2.write_html("figures/seasonality_DE.html")
fig2.show()

# Plot 3: Year-over-year comparison
yearly = daily.groupby(["year","month"])["avg_price_eur_mwh"].mean().reset_index()
fig3 = px.line(yearly, x="month", y="avg_price_eur_mwh", color="year",
               title="Monthly Avg Price by Year — Is This Winter Unusual?",
               labels={"avg_price_eur_mwh": "EUR/MWh", "month": "Month", "year": "Year"},
               markers=True, template="plotly_white")
fig3.update_xaxes(tickvals=list(range(1,13)),
                  ticktext=["Jan","Feb","Mar","Apr","May","Jun",
                            "Jul","Aug","Sep","Oct","Nov","Dec"])
fig3.write_html("figures/yoy_comparison_DE.html")
fig3.show()

# Summary stats
print(daily["avg_price_eur_mwh"].describe().round(2))
print("\nCorrelation with month (seasonality signal):",
      daily[["avg_price_eur_mwh","month"]].corr().iloc[0,1].round(3))

count    1971.00
mean      117.41
std        88.58
min       -53.87
25%        69.01
50%        94.34
75%       129.43
max       699.44
Name: avg_price_eur_mwh, dtype: float64

Correlation with month (seasonality signal): 0.173


In [52]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import joblib
import os

os.makedirs("models", exist_ok=True)
os.makedirs("figures", exist_ok=True)

daily = pd.read_csv("data/clean/prices_daily_DE.csv", parse_dates=["date"])
daily = daily.sort_values("date").reset_index(drop=True)

# Feature engineering
for n in [1, 2, 3, 7, 14, 30]:
    daily[f"lag_{n}"] = daily["avg_price_eur_mwh"].shift(n)

daily["rolling_7d_mean"]  = daily["avg_price_eur_mwh"].shift(1).rolling(7).mean()
daily["rolling_30d_mean"] = daily["avg_price_eur_mwh"].shift(1).rolling(30).mean()
daily["month"]            = daily["date"].dt.month
daily["dayofweek"]        = daily["date"].dt.dayofweek
daily["year"]             = daily["date"].dt.year

daily = daily.dropna().reset_index(drop=True)

FEATURES = [f"lag_{lag_n}" for lag_n in [1,2,3,7,14,30]] + \
           ["rolling_7d_mean", "rolling_30d_mean", "month", "dayofweek", "year"]
TARGET = "avg_price_eur_mwh"

# Train / test split
split_idx = len(daily) - 90
train = daily.iloc[:split_idx]
test  = daily.iloc[split_idx:]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Evaluate
preds = model.predict(X_test_scaled)
mae   = mean_absolute_error(y_test, preds)
rmse  = np.sqrt(mean_squared_error(y_test, preds))
r2    = model.score(X_test_scaled, y_test)
print(f"MAE:  {mae:.2f} EUR/MWh")
print(f"RMSE: {rmse:.2f} EUR/MWh")
print(f"R²:   {r2:.3f}")

# Plot: predicted vs actual
fig_pred = go.Figure()
fig_pred.add_trace(go.Scatter(x=test["date"], y=y_test.values,
                              mode="lines", name="Actual",
                              line=dict(color="steelblue", width=1.5)))
fig_pred.add_trace(go.Scatter(x=test["date"], y=preds,
                              mode="lines", name="Predicted",
                              line=dict(color="orange", width=1.5, dash="dash")))
fig_pred.update_layout(title="Linear Regression — Predicted vs Actual (Test Set, Germany)",
                       yaxis_title="EUR/MWh", xaxis_title="Date", template="plotly_white")
fig_pred.write_html("figures/lr_predictions_DE.html")
fig_pred.show()

# Coefficients
coef_df = pd.Series(model.coef_, index=FEATURES).sort_values().reset_index()
coef_df.columns = ["feature", "coefficient"]
fig_coef = px.bar(coef_df, x="coefficient", y="feature", orientation="h",
                  title="Linear Regression Coefficients",
                  labels={"coefficient": "Coefficient", "feature": "Feature"},
                  color_discrete_sequence=["steelblue"], template="plotly_white")
fig_coef.add_vline(x=0, line_color="black", line_width=0.8)
fig_coef.write_html("figures/lr_coefficients_DE.html")
fig_coef.show()

# Save model and scaler
joblib.dump(model,  "models/lr_price_forecast_DE.pkl")
joblib.dump(scaler, "models/lr_scaler_DE.pkl")

test_out = test[["date"]].copy()
test_out["actual"]    = y_test.values
test_out["predicted"] = preds
test_out.to_csv("data/clean/lr_test_predictions_DE.csv", index=False)
print("Model and predictions saved.")

# 30-day forecast from any date (plug into backend later)
def predict_from_date(start_date, daily_df, trained_model, fitted_scaler, features, days=30):
    """
    Predict electricity prices for 30 days starting from any date.
    If start_date is in the dataset, uses real historical prices as seed.
    If start_date is beyond the dataset, seeds from the last known row
    and iteratively forecasts forward to the start date first.
    """
    start_date = pd.Timestamp(start_date)

    df_copy = daily_df.copy()
    for n in [1, 2, 3, 7, 14, 30]:
        df_copy[f"lag_{n}"] = df_copy["avg_price_eur_mwh"].shift(n)
    df_copy["rolling_7d_mean"]  = df_copy["avg_price_eur_mwh"].shift(1).rolling(7).mean()
    df_copy["rolling_30d_mean"] = df_copy["avg_price_eur_mwh"].shift(1).rolling(30).mean()
    df_copy["month"]            = df_copy["date"].dt.month
    df_copy["dayofweek"]        = df_copy["date"].dt.dayofweek
    df_copy["year"]             = df_copy["date"].dt.year
    df_copy = df_copy.dropna().reset_index(drop=True)

    last_known_date = df_copy["date"].max()

    # Seed from matching row if date is in dataset
    if start_date <= last_known_date:
        idx = df_copy[df_copy["date"] <= start_date].index[-1]
        last_row = df_copy.iloc[idx].copy()
    else:
        # Forecast forward from last known date to reach start date
        last_row = df_copy.iloc[-1].copy()
        current_date = last_known_date

        while current_date < start_date - pd.Timedelta(days=1):
            X = pd.DataFrame([last_row[features]])
            X_scaled = fitted_scaler.transform(X)
            pred = trained_model.predict(X_scaled)[0]

            current_date += pd.Timedelta(days=1)
            last_row["lag_30"]    = last_row["lag_14"]
            last_row["lag_14"]    = last_row["lag_7"]
            last_row["lag_7"]     = last_row["lag_3"]
            last_row["lag_3"]     = last_row["lag_2"]
            last_row["lag_2"]     = last_row["lag_1"]
            last_row["lag_1"]     = pred
            last_row["date"]      = current_date
            last_row["month"]     = current_date.month
            last_row["dayofweek"] = current_date.dayofweek
            last_row["year"]      = current_date.year

    # Forecast 30 days from start date
    predictions = []
    for _ in range(days):
        X = pd.DataFrame([last_row[features]])
        X_scaled = fitted_scaler.transform(X)
        pred = trained_model.predict(X_scaled)[0]

        date = last_row["date"] + pd.Timedelta(days=1)
        predictions.append({
            "date": str(date.date()),
            "predicted_price_eur_mwh": round(pred, 2)
        })

        last_row["lag_30"]    = last_row["lag_14"]
        last_row["lag_14"]    = last_row["lag_7"]
        last_row["lag_7"]     = last_row["lag_3"]
        last_row["lag_3"]     = last_row["lag_2"]
        last_row["lag_2"]     = last_row["lag_1"]
        last_row["lag_1"]     = pred
        last_row["date"]      = date
        last_row["month"]     = date.month
        last_row["dayofweek"] = date.dayofweek
        last_row["year"]      = date.year

    return pd.DataFrame(predictions)

# Change this to any date you want
START_DATE = "2026-05-25"
forecast_df = predict_from_date(START_DATE, daily, model, scaler, FEATURES)
print(f"\n30-Day Forecast starting {START_DATE}:")
print(forecast_df.to_string(index=False))

# Plot the 30-day forecast
fig_forecast = go.Figure()
fig_forecast.add_trace(go.Scatter(x=forecast_df["date"], y=forecast_df["predicted_price_eur_mwh"],
                                  mode="lines+markers", name="Forecast",
                                  line=dict(color="steelblue", width=2),
                                  marker=dict(size=5)))
fig_forecast.update_layout(title=f"30-Day Electricity Price Forecast starting {START_DATE} (Germany)",
                            yaxis_title="EUR/MWh", xaxis_title="Date", template="plotly_white")
fig_forecast.write_html(f"figures/lr_forecast_{START_DATE}.html")
fig_forecast.show()

forecast_df.to_csv(f"data/clean/lr_forecast_{START_DATE}.csv", index=False)
print("Forecast saved.")

MAE:  23.09 EUR/MWh
RMSE: 29.56 EUR/MWh
R²:   0.265


Model and predictions saved.

30-Day Forecast starting 2026-05-25:
      date  predicted_price_eur_mwh
2026-05-26                   101.91
2026-05-27                   121.28
2026-05-28                   119.40
2026-05-29                   109.85
2026-05-30                    98.30
2026-05-31                    87.01
2026-06-01                    73.87
2026-06-02                   105.08
2026-06-03                   120.85
2026-06-04                   118.61
2026-06-05                   110.13
2026-06-06                    99.02
2026-06-07                    87.45
2026-06-08                    73.87
2026-06-09                   105.04
2026-06-10                   120.92
2026-06-11                   118.73
2026-06-12                   110.25
2026-06-13                    99.10
2026-06-14                    87.51
2026-06-15                    73.92
2026-06-16                   105.10
2026-06-17                   120.97
2026-06-18                   118.77
2026-06-19                   110.

Forecast saved.
